In [ ]:
# test_1.py
import pytest

from solution import GraphQL, SchemaError


def test_p1_constructs_with_valid_schema():
    schema = """
        type Query {
            hello: String
        }
        type User {
            id: ID
            name: String
        }
    """
    try:
        GraphQL(schema)
    except Exception as e:
        pytest.fail(f"GraphQL failed to construct with a valid schema: {e}")


def test_p1_get_type_returns_truthy_for_defined_object_type():
    schema = """
        type User {
            id: ID
            name: String
        }
    """
    g = GraphQL(schema)
    t = g.get_type("User")
    assert t, "get_type('User') must return a truthy value for defined types"


def test_p1_using_builtin_scalars_in_fields_is_allowed():
    schema = """
        type Query {
            a: String
            b: Int
            c: Float
            d: Boolean
            e: ID
        }
    """
    try:
        GraphQL(schema)
    except Exception as e:
        pytest.fail(f"Built-in scalars used in fields should not raise: {e}")


def test_p1_undefined_field_type_raises_schema_error():
    schema = """
        type Query {
            user: UnknownType
        }
    """
    with pytest.raises(SchemaError):
        GraphQL(schema)

In [ ]:
# test_2.py
import pytest

from solution import GraphQL, SchemaError


SCHEMA = """
    type Query {
        message: String
        user: User
    }

    type User {
        id: ID
        name: String
        address: Address
    }

    type Address {
        city: String
        zip: String
    }
"""


def test_p2_execute_simple_root_field():
    g = GraphQL(SCHEMA)
    query = "{ message }"
    data = {"message": "Hi"}
    result = g.execute(query, data)
    assert result == {"data": {"message": "Hi"}}


def test_p2_execute_nested_fields():
    g = GraphQL(SCHEMA)
    query = "{ user { name address { city } } }"
    data = {"user": {"name": "Alice", "address": {"city": "NYC"}}}
    result = g.execute(query, data)
    assert result == {"data": {"user": {"name": "Alice", "address": {"city": "NYC"}}}}


def test_p2_execute_multiple_root_fields():
    g = GraphQL(SCHEMA)
    query = "{ message user { id name } }"
    data = {"message": "Hello", "user": {"id": "1", "name": "Bob"}}
    result = g.execute(query, data)
    assert result == {"data": {"message": "Hello", "user": {"id": "1", "name": "Bob"}}}

In [ ]:
# test_3.py
import pytest

from solution import GraphQL, SchemaError


SCHEMA = """
    type Query {
        user(id: ID): User
    }
    type User {
        name: String
    }
"""


def test_p3_literal_argument_used_to_select_data():
    g = GraphQL(SCHEMA)
    # Implementation is free to choose how to "use" args.
    # This dataset requires using the 'id' argument to pick the right object.
    data = {
        "user": {
            "1": {"name": "Alice"},
            "2": {"name": "Bob"},
            "99": {"name": "Zed"},
        }
    }
    query = '{ user(id: "2") { name } }'
    result = g.execute(query, data)
    assert result == {"data": {"user": {"name": "Bob"}}}


def test_p3_argument_presence_does_not_break_other_fields():
    g = GraphQL(SCHEMA)
    data = {"user": {"1": {"name": "Alice"}}}
    query = '{ user(id: "1") { name } }'
    result = g.execute(query, data)
    assert result == {"data": {"user": {"name": "Alice"}}}

In [ ]:
# test_4.py
import pytest

from solution import GraphQL


SCHEMA = """
    type Query {
        user(id: ID): User
        message: String
    }
    type User {
        id: ID
        name: String
    }
"""


def test_p4_resolver_priority_over_root_value():
    resolvers = {"Query": {"message": lambda parent, args: "FromResolver"}}
    g = GraphQL(SCHEMA, resolvers)
    query = "{ message }"
    data = {"message": "FromRoot"}
    result = g.execute(query, data)
    assert result == {"data": {"message": "FromResolver"}}


def test_p4_resolver_receives_parent_and_args():
    calls = []

    def user_resolver(parent, args):
        calls.append((parent, dict(args)))
        return {"id": args.get("id"), "name": "Resolved"}

    resolvers = {"Query": {"user": user_resolver}}
    g = GraphQL(SCHEMA, resolvers)
    query = '{ user(id: "42") { id name } }'
    data = {"user": {"42": {"id": "42", "name": "Ignored"}}}
    result = g.execute(query, data)
    assert result["data"]["user"]["id"] == "42"
    assert result["data"]["user"]["name"] == "Resolved"
    assert calls and calls[0][0] == data and calls[0][1] == {"id": "42"}


def test_p4_nested_resolvers_are_supported():
    schema = """
        type Query { user: User }
        type User { name: String }
    """
    resolvers = {
        "Query": {"user": lambda parent, args: {"key": "Alice"}},
        "User": {"name": lambda parent, args: parent.get("key")},
    }
    g = GraphQL(schema, resolvers)
    result = g.execute("{ user { name } }", {})
    assert result == {"data": {"user": {"name": "Alice"}}}


def test_p4_resolver_returning_none_is_null():
    resolvers = {"Query": {"user": lambda parent, args: None}}
    g = GraphQL(SCHEMA, resolvers)
    result = g.execute("{ user { name } }", {})
    assert result == {"data": {"user": None}}

In [ ]:
# test_5.py
import pytest

from solution import GraphQL


SCHEMA = """
    type Query {
        user(id: ID): User
        posts(limit: Int): [Post]
    }
    type User { name: String }
    type Post { title: String }
"""


def test_p5_single_variable_substitution():
    resolvers = {
        "Query": {"user": lambda p, a: {"name": "Alice"} if a.get("id") == "1" else None}
    }
    g = GraphQL(SCHEMA, resolvers)
    query = 'query GetUser($uid: ID) { user(id: $uid) { name } }'
    result = g.execute(query, {}, variables={"uid": "1"})
    assert result == {"data": {"user": {"name": "Alice"}}}


def test_p5_multiple_variables_used_in_different_fields():
    resolvers = {
        "Query": {
            "user": lambda p, a: {"name": "Alice"} if a.get("id") == "1" else None,
            "posts": lambda p, a: [{"title": "A"}, {"title": "B"}][: int(a.get("limit") or 0)],
        }
    }
    g = GraphQL(SCHEMA, resolvers)
    query = 'query X($uid: ID, $lim: Int) { user(id: $uid) { name } posts(limit: $lim) { title } }'
    result = g.execute(query, {}, variables={"uid": "1", "lim": 1})
    assert result == {"data": {"user": {"name": "Alice"}, "posts": [{"title": "A"}]}}

In [ ]:
# test_6.py
import pytest

from solution import GraphQL, QueryError


SCHEMA = """
    type Query {
        user(id: ID!): User
        posts(tags: [String!]): [Post!]
    }
    type User { id: ID! name: String }
    type Post { title: String }
"""


def test_p6_parser_accepts_list_and_non_null_modifiers():
    try:
        GraphQL(SCHEMA)
    except Exception as e:
        pytest.fail(f"Schema with list/non-null should be accepted: {e}")


def test_p6_list_field_execution_with_literal_list_argument():
    resolvers = {
        "Query": {
            "posts": lambda p, a: (
                [{"title": "GQL"}, {"title": "Py"}] if "dev" in (a.get("tags") or []) else []
            )
        }
    }
    g = GraphQL(SCHEMA, resolvers)
    query = '{ posts(tags: ["dev", "news"]) { title } }'
    result = g.execute(query, {})
    assert result == {"data": {"posts": [{"title": "GQL"}, {"title": "Py"}]}}


def test_p6_missing_required_argument_raises_query_error():
    resolvers = {"Query": {"user": lambda p, a: {"id": "x"}}}
    g = GraphQL(SCHEMA, resolvers)
    query = "{ user { id } }"  # id: ID! not provided
    with pytest.raises(QueryError):
        g.execute(query, {})


def test_p6_null_variable_for_non_null_argument_raises_query_error():
    resolvers = {"Query": {"user": lambda p, a: {"id": "x"}}}
    g = GraphQL(SCHEMA, resolvers)
    query = "query Q($id: ID!) { user(id: $id) { id } }"
    with pytest.raises(QueryError):
        g.execute(query, {}, variables={"id": None})

In [ ]:
# test_7.py
import pytest

from solution import GraphQL


SCHEMA_NULLABLE_PARENT = """
    type Query {
        user: User
    }
    type User {
        name: String!
    }
"""

SCHEMA_NON_NULL_CHAIN = """
    type Query {
        user: User!
    }
    type User {
        address: Address!
    }
    type Address {
        city: String!
    }
"""


def test_p7_non_null_field_nullifies_nullable_parent_object():
    # name: String! -> resolver returns None -> user becomes null (Query.user is nullable)
    resolvers = {"Query": {"user": lambda p, a: {}}, "User": {"name": lambda p, a: None}}
    g = GraphQL(SCHEMA_NULLABLE_PARENT, resolvers)
    result = g.execute("{ user { name } }", {})
    # We don't require specific error shapes, only that it exists and data was nullified appropriately.
    assert result.get("data") == {"user": None}
    assert isinstance(result.get("errors"), list) and len(result["errors"]) > 0


def test_p7_error_bubbles_through_non_null_chain_to_null_data():
    # Query.user: User! -> User.address: Address! -> Address.city: String! returns None
    # This must nullify up to the nearest nullable boundary, which is the top-level data.
    resolvers = {
        "Query": {"user": lambda p, a: {}},
        "User": {"address": lambda p, a: {}},
        "Address": {"city": lambda p, a: None},
    }
    g = GraphQL(SCHEMA_NON_NULL_CHAIN, resolvers)
    result = g.execute("{ user { address { city } } }", {})
    assert result.get("data") is None
    assert isinstance(result.get("errors"), list) and len(result["errors"]) > 0

In [ ]:
# test_8.py
import pytest

from solution import GraphQL, QueryError


SCHEMA = """
    type Query {
        user: User
    }
    type User {
        id: ID
        name: String
        email: String
    }
"""


def test_p8_named_fragment_spread():
    g = GraphQL(SCHEMA)
    query = """
        query Q {
            user {
                ...UserFields
            }
        }
        fragment UserFields on User {
            id
            name
        }
    """
    data = {"user": {"id": "1", "name": "Alice", "email": "a@example.com"}}
    result = g.execute(query, data)
    assert result == {"data": {"user": {"id": "1", "name": "Alice"}}}


def test_p8_inline_fragment_with_type_condition():
    g = GraphQL(SCHEMA)
    query = """
        {
            user {
                ... on User {
                    name
                    email
                }
            }
        }
    """
    data = {"user": {"name": "Bob", "email": "b@example.com"}}
    result = g.execute(query, data)
    assert result == {"data": {"user": {"name": "Bob", "email": "b@example.com"}}}


def test_p8_spread_of_undefined_fragment_raises():
    g = GraphQL(SCHEMA)
    query = "{ user { ...Nope } }"
    with pytest.raises(QueryError):
        g.execute(query, {})

In [ ]:
# test_9.py
import pytest

from solution import GraphQL


SCHEMA = """
    interface Node { id: ID! }
    type User implements Node { id: ID! name: String }
    type Post implements Node { id: ID! title: String }

    union SearchResult = User | Post

    type Query {
        node(id: ID!): Node
        search(term: String): [SearchResult]
    }
"""


def test_p9_interface_resolution_with_inline_fragments():
    db = {
        "user-1": {"_type": "User", "id": "user-1", "name": "Alice"},
        "post-1": {"_type": "Post", "id": "post-1", "title": "GraphQL"},
    }
    resolvers = {
        "Query": {
            "node": lambda p, a: db.get(a.get("id")),
        },
        # implementors provide a type resolver for abstract types
        "Node": {"__resolve_type": lambda obj, info: obj.get("_type")},
    }
    g = GraphQL(SCHEMA, resolvers)
    query = """
        {
            node(id: "user-1") {
                id
                ... on User { name }
                ... on Post { title }
            }
        }
    """
    result = g.execute(query, {})
    assert result == {"data": {"node": {"id": "user-1", "name": "Alice"}}}


def test_p9_union_list_resolution_with_type_conditions():
    items = [
        {"_type": "User", "id": "u1", "name": "A"},
        {"_type": "Post", "id": "p1", "title": "T"},
    ]
    resolvers = {
        "Query": {"search": lambda p, a: items},
        "SearchResult": {"__resolve_type": lambda obj, info: obj.get("_type")},
    }
    g = GraphQL(SCHEMA, resolvers)
    query = """
        {
            search(term: "x") {
                ... on User { name }
                ... on Post { title }
            }
        }
    """
    result = g.execute(query, {})
    assert result == {"data": {"search": [{"name": "A"}, {"title": "T"}]}}

In [ ]:
# test_10.py
import pytest

from solution import GraphQL


SCHEMA = """
    type Query {
        user: User
    }
    type User {
        id: ID!
        name: String
        posts: [Post!]
    }
    type Post {
        title: String
    }
"""


def test_p10_schema_introspection_lists_known_types():
    g = GraphQL(SCHEMA)
    query = "{ __schema { types { name } } }"
    result = g.execute(query, {})
    assert "errors" not in result or not result["errors"]
    types = result.get("data", {}).get("__schema", {}).get("types", [])
    names = {t.get("name") for t in types if isinstance(t, dict)}
    # We only require core user-defined types to appear.
    assert {"Query", "User", "Post"}.issubset(names)


def test_p10_type_introspection_returns_basic_metadata():
    g = GraphQL(SCHEMA)
    query = """
        {
            __type(name: "User") {
                name
                kind
                fields {
                    name
                }
            }
        }
    """
    result = g.execute(query, {})
    t = result.get("data", {}).get("__type")
    assert isinstance(t, dict)
    assert t.get("name") == "User"
    assert t.get("kind") in ("OBJECT", "OBJECT".upper())  # allow flexible casing
    field_names = {f.get("name") for f in (t.get("fields") or []) if isinstance(f, dict)}
    assert {"id", "name", "posts"}.issubset(field_names)